In [17]:
!pip install requests beautifulsoup4 pandas

In [18]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from collections import deque

In [19]:
API_URL = "https://en.wikipedia.org/w/api.php"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Educational Web Scraping Project)"
}

In [20]:
def get_category_members(category):
    members = []
    continuation = {}

    while True:
        params = {
            "action": "query",
            "format": "json",
            "list": "categorymembers",
            "cmtitle": category,
            "cmtype": "page|subcat",
            "cmlimit": 500,
            **continuation
        }

        response = requests.get(
            API_URL,
            params=params,
            headers=HEADERS,
            timeout=30
        )

        response.raise_for_status()
        data = response.json()

        members.extend(data["query"]["categorymembers"])

        if "continue" not in data:
            break

        continuation = {
            "cmcontinue": data["continue"]["cmcontinue"]
        }

    return members

In [21]:
import sys

# Override the global HEADERS with a policy-compliant User-Agent to prevent 429 errors
HEADERS["User-Agent"] = "EducationalWebScraper/1.0 (contact: user@example.com) Python-requests"

queue = deque(["Category:Novels"])
visited_categories = set()
novel_titles = set()

while queue and len(novel_titles) < 1300:
    category = queue.popleft()

    if category in visited_categories:
        continue

    visited_categories.add(category)

    # Attempt to fetch category members with retry backoff for rate limits
    members = []
    retries = 5
    delay = 2
    success = False

    for attempt in range(retries):
        try:
            members = get_category_members(category)
            success = True
            break
        except Exception as e:
            if "429" in str(e):
                print(f"Rate limit (429) hit. Retrying {category} in {delay} seconds (Attempt {attempt+1}/{retries})...")
                time.sleep(delay)
                delay *= 2
            else:
                print(f"Failed to fetch {category}: {e}")
                time.sleep(2)
                break

    if not success:
        continue

    for item in members:
        title = item["title"]
        namespace = item["ns"]

        if namespace == 0:
            if not title.startswith(("List of ", "Outline of ")):
                novel_titles.add(title)

        elif namespace == 14:
            if title not in visited_categories:
                queue.append(title)

    print("Categories:", len(visited_categories), "| Novels:", len(novel_titles))

    # Add a polite 1-second delay between category fetches to respect Wikipedia's rate limit
    time.sleep(1)

print("Total novel pages:", len(novel_titles))

Categories: 1 | Novels: 1
Categories: 2 | Novels: 5
Categories: 3 | Novels: 5
Categories: 4 | Novels: 5
Categories: 5 | Novels: 5
Categories: 6 | Novels: 5
Categories: 7 | Novels: 6
Categories: 8 | Novels: 6
Categories: 9 | Novels: 6
Categories: 10 | Novels: 6
Categories: 11 | Novels: 6
Categories: 12 | Novels: 6
Categories: 13 | Novels: 6
Categories: 14 | Novels: 7
Categories: 15 | Novels: 7
Categories: 16 | Novels: 8
Categories: 17 | Novels: 12
Categories: 18 | Novels: 12
Categories: 19 | Novels: 15
Categories: 20 | Novels: 44
Categories: 21 | Novels: 45
Categories: 22 | Novels: 51
Categories: 23 | Novels: 60
Categories: 24 | Novels: 122
Categories: 25 | Novels: 139
Categories: 26 | Novels: 185
Categories: 27 | Novels: 248
Categories: 28 | Novels: 248
Categories: 29 | Novels: 271
Categories: 30 | Novels: 278
Categories: 31 | Novels: 282
Categories: 32 | Novels: 328
Categories: 33 | Novels: 332
Categories: 34 | Novels: 332
Categories: 35 | Novels: 332
Categories: 36 | Novels: 332
Cate

In [22]:
def extract_novel(title):
    url = "https://en.wikipedia.org/wiki/" + title.replace(" ", "_")

    response = requests.get(
        url,
        headers=HEADERS,
        timeout=30
    )

    if response.status_code != 200:
        return None

    soup = BeautifulSoup(response.text, "html.parser")

    title_tag = soup.find("h1")

    if not title_tag:
        return None

    novel_title = title_tag.get_text(strip=True)

    author = ""
    year = ""

    infobox = soup.find("table", class_="infobox")

    if infobox:
        for row in infobox.find_all("tr"):
            cells = row.find_all(["th", "td"])

            if len(cells) >= 2:
                label = cells[0].get_text(" ", strip=True).lower()
                value = cells[1].get_text(" ", strip=True)

                if "author" in label or "authors" in label:
                    author = value

                if "publication date" in label or "published" in label:
                    year = value

    description = ""

    content = soup.find("div", class_="mw-parser-output")

    if content:
        for paragraph in content.find_all("p"):
            text = paragraph.get_text(" ", strip=True)

            if text:
                description = text
                break

    return {
        "Title": novel_title,
        "Author": author,
        "Year": year,
        "Description": description,
        "URL": url
    }

In [23]:
novels = []

for i, title in enumerate(novel_titles):
    if len(novels) >= 1000:
        break

    try:
        data = extract_novel(title)

        if data:
            novels.append(data)

        if (i + 1) % 50 == 0:
            print("Processed:", i + 1, "| Collected:", len(novels))

        time.sleep(0.1)

    except Exception:
        continue

print("Total collected novels:", len(novels))

Processed: 50 | Collected: 50
Processed: 100 | Collected: 100
Processed: 150 | Collected: 150
Processed: 200 | Collected: 200
Processed: 250 | Collected: 250
Processed: 300 | Collected: 300
Processed: 350 | Collected: 350
Processed: 400 | Collected: 400
Processed: 450 | Collected: 450
Processed: 500 | Collected: 500
Processed: 550 | Collected: 550
Processed: 600 | Collected: 600
Processed: 650 | Collected: 650
Processed: 700 | Collected: 700
Processed: 750 | Collected: 750
Processed: 800 | Collected: 800
Processed: 850 | Collected: 850
Processed: 900 | Collected: 900
Processed: 950 | Collected: 950
Processed: 1000 | Collected: 999
Total collected novels: 1000


In [27]:
from IPython.display import display

display(df.head(10))

,Title,Author,Year,Description,URL
0,My Father's Wives,José Eduardo Agualusa,,My Father's Wives is a novel by the Angolan au...,https://en.wikipedia.org/wiki/My_Father's_Wives
1,Young adult romance literature,,,Young adult romance literature is a genre of b...,https://en.wikipedia.org/wiki/Young_adult_roma...
2,The Starlight Barking,Dodie Smith,1967,The Starlight Barking is a 1967 children's nov...,https://en.wikipedia.org/wiki/The_Starlight_Ba...
3,Khusi,Vijay Kumar Pandey,2014,Khusi ( Nepali : खुसी ) is an autobiographical...,https://en.wikipedia.org/wiki/Khusi
4,The Shadow over Innsmouth,H. P. Lovecraft,April 1936,The Shadow over Innsmouth is a horror novella ...,https://en.wikipedia.org/wiki/The_Shadow_over_...
5,Payback(novel),,,Payback ( German : Vergeltung ) is the second ...,https://en.wikipedia.org/wiki/Payback_(novel)
6,Novelist,,,A novelist is an author or writer of novels . ...,https://en.wikipedia.org/wiki/Novelist
7,Hop-Çiki-Yaya,,,Hop-Çiki-Yaya (AKA: Turkish Delight Mystery ) ...,https://en.wikipedia.org/wiki/Hop-Çiki-Yaya
8,Bram Stoker Award for Best Young Adult Novel,,,The Bram Stoker Award for Best Young Adult Nov...,https://en.wikipedia.org/wiki/Bram_Stoker_Awar...
9,The Wolves of Eternity,Karl Ove Knausgård,2021,The Wolves of Eternity ( Ulvene fra evighetens...,https://en.wikipedia.org/wiki/The_Wolves_of_Et...


In [29]:
from google.colab import files
import os

# Trigger the download directly. Ensure steps 1-5 have been run first to create the CSV file.
if os.path.exists("wikipedia_1000_novels.csv"):
    files.download("wikipedia_1000_novels.csv")
else:
    print("File not found on disk. Please re-run the previous scraping cells to generate the dataset first!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>